In [2]:
"""
DAV Lab 6: Feature Engineering
Normalization, Standardization, and Principal Component Analysis (PCA)

Install dependencies:
    pip install numpy pandas scikit-learn
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ------------------------------------------------------------
# Part 1: Student performance dataset for scaling
# ------------------------------------------------------------
data = pd.DataFrame(
    {
        "StudyHours": [2, 4, 6, 8, 10],
        "Attendance": [60, 70, 80, 90, 100],
        "PreviousMarks": [55, 65, 75, 85, 95],
        "AssignmentMarks": [50, 60, 70, 80, 90],
    },
    index=["S1", "S2", "S3", "S4", "S5"],
)

print("Original data:")
print(data)

# ------------------------------------------------------------
# Part 2: Min-Max normalization (each feature is mapped to [0, 1])
# Formula: (x - minimum) / (maximum - minimum)
normalized_data = (data - data.min()) / (data.max() - data.min())
print("\nMin-Max normalized data (manual):")
print(normalized_data.round(6))

# Equivalent scikit-learn implementation
minmax_scaler = MinMaxScaler()
normalized_sklearn = pd.DataFrame(
    minmax_scaler.fit_transform(data), columns=data.columns, index=data.index
)
print("\nMin-Max normalized data (scikit-learn):")
print(normalized_sklearn.round(6))

# ------------------------------------------------------------
# Part 3: Standardization (population standard deviation, ddof=0)
# Formula: z = (x - mean) / standard deviation
means = data.mean()
population_std = data.std(ddof=0)
standardized_data = (data - means) / population_std
print("\nMeans:")
print(means)
print("\nPopulation standard deviations:")
print(population_std)
print("\nStandardized data (manual):")
print(standardized_data.round(6))

# Equivalent implementation matching sklearn StandardScaler
standard_scaler = StandardScaler()
standardized_sklearn = pd.DataFrame(
    standard_scaler.fit_transform(data), columns=data.columns, index=data.index
)
print("\nStandardized data (scikit-learn):")
print(standardized_sklearn.round(6))

# ------------------------------------------------------------
# Part 4: Manual PCA demonstration with the two features in the slides
pca_data = pd.DataFrame(
    {
        "StudyHours": [1, 2, 3, 4],
        "PreviousMarks": [2, 3, 5, 4],
    },
    index=["S1", "S2", "S3", "S4"],
)
print("\nPCA input data:")
print(pca_data)

# Standardize using population standard deviation, as StandardScaler does.
pca_mean = pca_data.mean()
pca_std = pca_data.std(ddof=0)
Z = (pca_data - pca_mean) / pca_std
print("\nStandardized PCA data (Z):")
print(Z.round(6))

# Sample covariance matrix (divide by n - 1)
covariance_matrix = np.cov(Z.to_numpy(), rowvar=False, ddof=1)
print("\nCovariance matrix:")
print(pd.DataFrame(covariance_matrix, index=pca_data.columns, columns=pca_data.columns).round(6))

# Eigenvalues/eigenvectors of a symmetric covariance matrix.
# Sort from largest eigenvalue to smallest, and keep columns in matching order.
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

print("\nEigenvalues (largest first):")
print(eigenvalues.round(6))
print("\nEigenvectors (columns correspond to PC1, PC2):")
print(pd.DataFrame(eigenvectors, index=pca_data.columns, columns=["PC1", "PC2"]).round(6))

# Explained variance for each component and cumulative explained variance.
explained_variance_ratio = eigenvalues / eigenvalues.sum()
cumulative_explained_variance = np.cumsum(explained_variance_ratio)
print("\nExplained variance ratio:")
print(explained_variance_ratio.round(6))
print("\nCumulative explained variance:")
print(cumulative_explained_variance.round(6))

# Project standardized observations into the principal component space.
principal_components = Z.to_numpy() @ eigenvectors
pca_df = pd.DataFrame(principal_components, index=pca_data.index, columns=["PC1", "PC2"])
print("\nPrincipal components (PC1 and PC2):")
print(pca_df.round(6))

# Keep only the first principal component to reduce 2 features to 1.
reduced_df = pca_df[["PC1"]]
print("\nReduced dataset (one principal component):")
print(reduced_df.round(6))


Original data:
    StudyHours  Attendance  PreviousMarks  AssignmentMarks
S1           2          60             55               50
S2           4          70             65               60
S3           6          80             75               70
S4           8          90             85               80
S5          10         100             95               90

Min-Max normalized data (manual):
    StudyHours  Attendance  PreviousMarks  AssignmentMarks
S1        0.00        0.00           0.00             0.00
S2        0.25        0.25           0.25             0.25
S3        0.50        0.50           0.50             0.50
S4        0.75        0.75           0.75             0.75
S5        1.00        1.00           1.00             1.00

Min-Max normalized data (scikit-learn):
    StudyHours  Attendance  PreviousMarks  AssignmentMarks
S1        0.00        0.00           0.00             0.00
S2        0.25        0.25           0.25             0.25
S3        0.50        0.